# Installations and Imported Packages

In [ ]:
# !pip install -q transformers datasets wandb librosa torchaudio torchvision

In [ ]:
import os
import time
import random
import glob

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
import wandb

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score

from pathlib import Path
from tqdm import tqdm

from transformers import AutoFeatureExtractor, ASTForAudioClassification

import warnings
warnings.filterwarnings("ignore")

print(time.ctime())
print("All libraries are imported")

# User Defined Control Variable

Setting the flag variable **execute_milestone_solutions_code** as **False** ignore the exection of the cells containing the milestone solution code

In [ ]:
execute_milestone_solutions_code=False
print(execute_milestone_solutions_code)

# Weights & Biases Setup and Test run

In [ ]:
# try:
#     wandb.login()
#     print("Login Successful")
# except Exception as e:
#     print(f"Login Failed: {e}")

In [ ]:
# for run in range(5):
#     # Start a new wandb run to track this script.
#     run = wandb.init(
#         # Set the wandb entity where your project will be logged (generally your team name).
#         entity="21f2000660-dl-gen-ai-project-26-t1",
#         # Set the wandb project where this run will be logged.
#         project="21f2000660-t12026",
#         # Track hyperparameters and run metadata.
#         config={
#             "learning_rate": 0.02,
#             "architecture": "CNN",
#             "dataset": "CIFAR-100",
#             "epochs": 10,
#         },
#     )

# # Simulate training.
# epochs = 10
# offset = random.random() / 5
# for epoch in range(2, epochs):
#     acc = 1 - 2**-epoch - random.random() / epoch - offset
#     loss = 2**-epoch + random.random() / epoch + offset

#     # Log metrics to wandb.
#     run.log({"acc": acc, "loss": loss})

# # Finish the run and upload any remaining data.
# run.finish()

# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
DATA_ROOT = path # Enter dataset path

with os.scandir(path) as entries:
    directories = [entry.name for entry in entries]

GENRES = sorted(directories) # Make the list of all genres available (alphabetical order)
print(GENRES)

STEMS = {'drums':'drums.wav', 'vocals':'vocals.wav', 'bass':'bass.wav', 'other':'other.wav'} # Write here stems file name
print(STEMS)

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    #print(train_dataset)

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

        # Iterate through Genres
        # Check: if genre folder exists
        # CHECK : Completeness (Does it have all stems?)
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

    # Limits given in the questions
    limit_1 = 4 * 1024
    limit_2 = 5.0491 * 1024 * 1024
    limit_3 = 5.0493 * 1024 * 1024

    corrupted_songs = 0
    small_songs = 0
    large_songs = 0
    
    # Helper function to populate dict
    def add_to_dict(target_dict, genre, song_paths):
        for song_path in song_paths:
            for stem_key, stem_file in STEMS.items():
                target_dict[genre][stem_key].append(os.path.join(song_path, stem_file))

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        
        # Check: if genre folder exists
        if not os.path.isdir(genre_path):
            continue
            
        # Get all song folders
        song_folders = sorted([f.path for f in os.scandir(genre_path) if f.is_dir()])
        valid_songs = []

        #print(song_folders[:3])

        for song in song_folders:
            is_valid = True
            
            # All stems for a song should be available, if not invalid 
            for stem_file in STEMS.values():
                file_path = os.path.join(song, stem_file)
                
                if not os.path.exists(file_path):
                    is_valid = False
                    continue
                
                file_size = os.path.getsize(file_path)
                
                if file_size < limit_1:
                    corrupted_songs += 1
                    is_valid = False
                
                if file_size < limit_2:
                    small_songs += 1
                    
                if file_size > limit_3:
                    large_songs += 1

            if is_valid:
                valid_songs.append(song)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        
        split_index = int(len(valid_songs) * (1 - val_split))
        train_paths = valid_songs[:split_index]
        val_paths = valid_songs[split_index:]
        
        add_to_dict(train_dataset, genre, train_paths)
        add_to_dict(val_dataset, genre, val_paths)

    print('Q1',corrupted_songs + small_songs)
    print('Q2',abs(large_songs - small_songs))
    print('Q3',abs(len(train_dataset['reggae']['drums']) - len(val_dataset['country']['vocals'])))

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

#print(val[GENRE_TO_TEST])

In [ ]:
if execute_milestone_solutions_code:
    
    def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
        """
        Input:
            dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
        Output:
            df: Pandas DataFrame containing details of all files with silence >= 5s
        """
        records = []
        # ------------------- write your code here -------------------------------
    
        total_files = 0    # ---- COUNT TOTAL FILES ----
    
    
    
            # Load Audio
    
            # Find Non-Silent Intervals
    
            # CASE A: Fully silent
            # CASE B: START silence
            # CASE C: END silence
            # CASE D: MIDDLE silence
    
            # Store result
            # if max_silence >= threshold_sec:
            #     records.append({
            #         "Genre": genre,
            #         "Stem": stem_name,
            #         "Duration": round(total_duration, 2),
            #         "Max_Silence_Sec": round(max_silence, 2),
            #         "Silence_Location": ", ".join(silence_type),
            #         "File_Path": file_path
            #     })
        #-------------------------------------------------------------------------
    
        # Iterate through the nested dictionary
        for genre, stems in dataset_dict.items():
            for stem_name, file_paths in stems.items():
                for file_path in file_paths:
                    total_files += 1
                    
                    # Load Audio
                    try:
                        y, sr = librosa.load(file_path, sr=sr)
                        total_duration = librosa.get_duration(y=y, sr=sr)
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
                        continue
    
                    # Find Non-Silent Intervals
                    # intervals is a list of [start, end] sample indices of NON-silence
                    intervals = librosa.effects.split(y, top_db=top_db)
                    
                    silence_durations = []
                    silence_type = []
    
                    # CASE A: Fully silent
                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("Full")
                    else:
                        # CASE B: START silence (from 0 to first non-silent start)
                        if intervals[0][0] > 0:
                            start_silence = intervals[0][0] / sr
                            silence_durations.append(start_silence)
                            if start_silence >= threshold_sec:
                                 silence_type.append("Start")
    
                        # CASE C: END silence (from last non-silent end to total length)
                        if intervals[-1][1] < len(y):
                            end_silence = (len(y) - intervals[-1][1]) / sr
                            silence_durations.append(end_silence)
                            if end_silence >= threshold_sec:
                                 silence_type.append("End")
    
                        # CASE D: MIDDLE silence (gaps between intervals)
                        for i in range(len(intervals) - 1):
                            silence_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                            silence_durations.append(silence_gap)
                            if silence_gap >= threshold_sec:
                                 silence_type.append("Middle")
                        
                        # Determine max silence for this file
                        if silence_durations:
                            max_silence = max(silence_durations)
                        else:
                            max_silence = 0.0
    
                    # Store result
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 4),
                            "Silence_Location": ", ".join(set(silence_type)), # use set to avoid duplicates
                            "File_Path": file_path
                        })
        df = pd.DataFrame(records)
        return df
    
    
    # --- EXECUTION ---
    # Pass your 'tr' (training) dictionary here.
    # Ensure 'tr' is defined from your previous build_dataset code.
    df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
    
    # --- RESULTS ANALYSIS ---
    
    # ------------------- write your code here -------------------------------
    
    print('Q4',len(df_silence))
    print('Q5',len(df_silence[df_silence['Stem'] == 'vocals']))
    print('Q6',df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean())
    print('Q7',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]))
    print('Q8',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Silence_Location'] == 'Middle')]))
    print('Q9',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Max_Silence_Sec'] >= 10.0)]))
    #-------------------------------------------------------------------------
    # Hint: Create a pivot Table: Count by Genre vs Stem
    
    pivot_table = df_silence.pivot_table(index='Genre', columns='Stem', values='Max_Silence_Sec', aggfunc='count', fill_value=0)
    print("\n--- Pivot Table ---")
    print(pivot_table)

In [ ]:
if execute_milestone_solutions_code:
    
    stems_audio = []
    try:
        for key in STEM_KEYS:
        # ------------------- write your code here -------------------------------
        # Load audio (Duration 5.0s for speed/consistency)
            file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
    
            y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
            stems_audio.append(y)
        #-------------------------------------------------------------------------
    
        print("Audio loaded successfully.")
    except NameError:
        print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
    except IndexError:
        print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
if execute_milestone_solutions_code:
    
    # ------------------- write your code here -------------------------------
    # Stack them into a numpy array (Shape: 4 x Samples)
    stems_stack = np.vstack(stems_audio)
    
    # Mix the stems by summing them element-wise
    mix_raw = np.sum(stems_stack, axis=0)
    
    # Calculate RMS Amplitude MANUALLY
    rms_val = np.sqrt(np.mean(mix_raw**2))
    
    #Peak Normalization
    max_val = np.max(np.abs(mix_raw))
    
    if max_val > 0:
        mix_norm = mix_raw / max_val
    else:
        mix_norm = mix_raw
    
    # VALIDATION
    assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
    #------------------------------------------------------------------------
    
    print('Q10',len(mix_raw))
    print('Q11',rms_val)
    print('Q12',max_val)

# Milestone 2

In [ ]:
if execute_milestone_solutions_code:
    
    #Mean duration of Jazz stems

    jazz_durations = []
    for stem_dict in tr.get('jazz').values():
        for file_path in stem_dict:
            dur = librosa.get_duration(path=file_path)
            jazz_durations.append(dur)
    
    mean_jazz_dur = np.mean(jazz_durations)
    print('Q1',mean_jazz_dur)

In [ ]:
if execute_milestone_solutions_code:

    #Unique sample rates
    
    unique_srs = set()
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                file_sr = librosa.get_samplerate(file_path)
                unique_srs.add(file_sr)
    
    print('Q2', unique_srs)

In [ ]:
if execute_milestone_solutions_code:
    
    #Corrupted or zero-byte files
    
    zero_byte_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                if os.path.getsize(file_path) == 0:
                    zero_byte_count += 1
    
    print('Q3',zero_byte_count)

In [ ]:
if execute_milestone_solutions_code:
    
    #Average peak amplitude for vocal stems
    
    vocal_peak_dbs = []
    for genre, stems in tr.items():
        vocal_files = stems.get('vocals')
        for file_path in vocal_files:
            y, sr = librosa.load(file_path, sr=SR)
            peak_amp = np.max(np.abs(y))
            peak_db = 20 * np.log10(max(peak_amp, 1e-10))
            vocal_peak_dbs.append(peak_db)
    
    avg_vocal_peak_db = np.mean(vocal_peak_dbs)
    print('Q4',avg_vocal_peak_db)

In [ ]:
if execute_milestone_solutions_code:
    
    #Spectral Centroids
    
    genre_mean_centroids = {}
    
    for genre, stems in tr.items():
        genre_centroids = []
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y, _ = librosa.load(file_path, sr=SR)
                # Compute spectral centroid
                sc = librosa.feature.spectral_centroid(y=y, sr=SR)
                # Take the mean centroid of this specific file
                genre_centroids.append(np.mean(sc))
                
        if genre_centroids:
            genre_mean_centroids[genre] = np.mean(genre_centroids)
    
    print(genre_mean_centroids)
    
    blues_centroid = genre_mean_centroids.get('blues')
    print('Q5',blues_centroid)
    
    highest_sc_genre = max(genre_mean_centroids, key=genre_mean_centroids.get)
    print('Q6',highest_sc_genre)

In [ ]:
if execute_milestone_solutions_code:
    
    #Silence in the first 0.5 seconds
    silence_count = 0
    for genre, stems in tr.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                y_start, sr = librosa.load(file_path, sr=SR, duration=0.5)
                
                intervals = librosa.effects.split(y_start, top_db=TOP_DB)
                
                if len(intervals) == 0 or intervals[0][0] > 0:
                    silence_count += 1
    
    print('Q7',silence_count)

# Programming Question

Complete the code as instructed in comments and answer the questions that follow.

In [ ]:
# --- 1. Setup and Preprocessing ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# --- 3. Model Training (Decision Tree) ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

'''
YOUR CODE HERE

y_pred = # COMPUTE PREDICTED VALUES
macro_f1 = # COMPUTE VALIDATION MACRO F1 SCORE
cm = # COMPUTE CONFUSION MATRIX
cr = # COMPUTE CLASSIFICATION REPORT

'''

y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred, target_names=GENRES)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

In [ ]:
if execute_milestone_solutions_code:
    
    '''
    YOUR CODE HERE
    
    Visualize the confusion matrix and compute TP, TN, FP, FN for all genres.
    '''
    # Visualize Confusion Matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
    plt.title('Confusion Matrix: Decision Tree')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    print('\n')
    print('--- TP, TN, FP, FN for all genres  ---')
    for i,genre in enumerate(GENRES):
        tp=cm[i, i]
        fp=cm[:, i].sum()-tp
        fn=cm[i, :].sum()-tp
        tn=cm.sum()-(tp+fp+fn)
            
        print(genre,"\t\t",tp,tn,fp,fn)

# Milestone 4

## Provided Helper Functions

In [ ]:
# directory paths

INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

In [ ]:
# provided helper function 1

def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

In [ ]:
# provided helper function 2

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

In [ ]:
# provided helper function 3

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

In [ ]:
waveform, sr = torchaudio.load('/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav')
print("Q2", tuple(waveform.shape))

In [ ]:
pf_tensor = torch.load('/kaggle/working/features/train/blues/mashup_000.pt')
print("Q3",tuple(pf_tensor.shape))

In [ ]:
class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: Define the CNN Backbone using nn.Sequential
        # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        # TODO 2: Define the RNN Component
        # Hint: Calculate the flattened feature size coming out of your CNN.
        # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
        # Channels = 64. So, Input Size = Channels * Mels = 2048.
        # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
        self.lstm = nn.LSTM(
            input_size=2048, 
            hidden_size=64, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        
        # TODO 3: Define the Classifier
        # Create a Fully Connected (Linear) layer. 
        # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.
        self.fc = nn.Linear(in_features=128, out_features=num_classes)

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        
        # TODO 4: Pass 'x' through the CNN backbone
        # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)
        x = self.cnn(x)
        
        # TODO 5: Bridge the gap (Shape Manipulation)
        # Permute and reshape the CNN output to be compatible with the LSTM.
        # Extract b, c, f, t from the tensor shape.
        # Permute the dimensions to bring Time forward, then reshape to flatten Channels and Mels.
        # Target shape for LSTM: (Batch, Time_Steps, Channels * Mels)
        b, c, f, t = x.shape
        print("Q4",tuple(x.shape))
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)
        
        # TODO 6: Pass the reshaped sequence through the LSTM
        # The LSTM returns output and (hidden_state, cell_state). You only need the output.
        lstm_out, _ = self.lstm(x)
        
        # TODO 7: Global Max Pooling over the time dimension
        # Collapse the sequence down to a single vector using torch.max() over the time dimension (dim=1).
        # Note: torch.max returns both values and indices. You only need the values.
        pooled_out, _ = torch.max(lstm_out, dim=1)
        
        # TODO 8: Pass the pooled vector through the fully connected layer
        logits = self.fc(pooled_out)
        
        return logits

In [ ]:
crnn_model = CRNN(num_classes=10)

pf_tensor_dataset = PrecomputedFeatureDataset(OUTPUT_DIR)
train_loader = DataLoader(pf_tensor_dataset, batch_size=32)
real_features, real_labels = next(iter(train_loader))
dummy_input = torch.randn(32, 1, 128, 1292)
crnn_model(dummy_input)

lstm_params = sum(p.numel() for p in temp_model.lstm.parameters() if p.requires_grad)
print("Q5",lstm_params)

# Milestone 5

In [ ]:
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=100)

In [ ]:
file_path = '/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav'
y, sr = librosa.load(file_path, sr=16000, duration=10)
print(f"Q2: {y.shape}")

In [ ]:
checkpoint = "MIT/ast-finetuned-audioset-10-10-0.4593"
extractor = AutoFeatureExtractor.from_pretrained(checkpoint)

# dummy audio as per the question
dummy_mix = np.ones(160000)
inputs = extractor(dummy_mix, sampling_rate=16000, return_tensors="pt")

# Squeeze(0)
ast_input = inputs['input_values'].squeeze(0)
print(f"Q3: {ast_input.shape}")

In [ ]:
model = ASTForAudioClassification.from_pretrained(
    checkpoint,
    num_labels=10,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Q4 : {total_params}")

In [ ]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])
y = y_test / (np.max(np.abs(y_test)) + 1e-9)
print(f"Q5 : {y}")

# End to end pipeline

In [ ]:
# --- CONFIGURATION ---
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'
STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

GENRES = sorted(["blues", "classical", "country", "disco", "hiphop", 
                 "jazz", "metal", "pop", "reggae", "rock"])
SR = 16000
DURATION = 10
TARGET_SAMPLES = SR * DURATION
SAMPLES_PER_SET = 20 # 20 files * 5 sets = 100 files per genre

# --- 1. STRICT VALIDATION ENGINE ---
def load_and_pad(path):
    """Safely loads, converts to mono, pads/truncates to exact length."""
    waveform, sr = torchaudio.load(path)
    if sr != SR:
        waveform = T.Resample(sr, SR)(waveform)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        
    if waveform.shape[1] > TARGET_SAMPLES:
        waveform = waveform[:, :TARGET_SAMPLES]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, TARGET_SAMPLES - waveform.shape[1]))
    return waveform

def validate_data(stems_path, noise_path):
    print("Validating all stems and noise files. This ensures no corrupted mashups...")
    valid_stems = {g: [] for g in GENRES}
    
    # Validate Stems
    for genre in GENRES:
        folders = glob.glob(os.path.join(stems_path, genre, '*'))
        for folder in folders:
            is_valid = True
            for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
                p = os.path.join(folder, stem)
                if not os.path.exists(p) or os.path.getsize(p) < 1000:
                    is_valid = False; break
            if is_valid: valid_stems[genre].append(folder)
            
    # Validate Noise
    raw_noise = glob.glob(os.path.join(noise_path, '**', '*.wav'), recursive=True)
    valid_noise = [n for n in raw_noise if os.path.getsize(n) > 1000]
    
    print(f"Validation complete! Found valid noise files: {len(valid_noise)}")
    for g in GENRES: print(f"  {g}: {len(valid_stems[g])} valid songs")
    return valid_stems, valid_noise

# --- 2. THE 5-SET MASHUP GENERATOR ---
def generate_5_set_dataset(valid_stems, valid_noise, output_dir, samples_per_set=20):
    os.makedirs(output_dir, exist_ok=True)
    
    for genre in GENRES:
        print(f"\nGenerating 5-Set Curriculum for: {genre.upper()}")
        genre_out = os.path.join(output_dir, genre)
        os.makedirs(genre_out, exist_ok=True)
        
        songs = valid_stems[genre]
        if len(songs) < 4:
            print(f"Skipping {genre}, not enough valid data.")
            continue
            
        for i in tqdm(range(samples_per_set)):
            
            # Helper to mix and save
            def save_mix(stems_dict, use_noise, filename):
                mix = torch.zeros((1, TARGET_SAMPLES))
                for path in stems_dict.values():
                    mix += load_and_pad(path)
                
                if use_noise and valid_noise:
                    n_path = random.choice(valid_noise)
                    n_wave = load_and_pad(n_path)
                    mix += (n_wave * random.uniform(0.1, 0.3))
                
                # Normalize to prevent clipping
                mix = mix / (torch.max(torch.abs(mix)) + 1e-9)
                torchaudio.save(os.path.join(genre_out, filename), mix, SR)

            # ---------------------------------------------------------
            # SET 1: Same Stems, No Noise
            # ---------------------------------------------------------
            s1_folder = random.choice(songs)
            s1_stems = {s: os.path.join(s1_folder, s) for s in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']}
            save_mix(s1_stems, use_noise=False, filename=f"set1_clean_{i:03d}.wav")

            # ---------------------------------------------------------
            # SET 2: Same Stems, With Noise
            # ---------------------------------------------------------
            s2_folder = random.choice(songs)
            s2_stems = {s: os.path.join(s2_folder, s) for s in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']}
            save_mix(s2_stems, use_noise=True, filename=f"set2_noisy_{i:03d}.wav")

            # ---------------------------------------------------------
            # SET 3: Intra-Genre Mashup, No Noise
            # ---------------------------------------------------------
            s3_folders = random.sample(songs, 4)
            s3_stems = {
                'drums.wav': os.path.join(s3_folders[0], 'drums.wav'),
                'bass.wav': os.path.join(s3_folders[1], 'bass.wav'),
                'vocals.wav': os.path.join(s3_folders[2], 'vocals.wav'),
                'other.wav': os.path.join(s3_folders[3], 'other.wav')
            }
            save_mix(s3_stems, use_noise=False, filename=f"set3_mashclean_{i:03d}.wav")

            # ---------------------------------------------------------
            # SET 4: Intra-Genre Mashup, With Noise
            # ---------------------------------------------------------
            s4_folders = random.sample(songs, 4)
            s4_stems = {
                'drums.wav': os.path.join(s4_folders[0], 'drums.wav'),
                'bass.wav': os.path.join(s4_folders[1], 'bass.wav'),
                'vocals.wav': os.path.join(s4_folders[2], 'vocals.wav'),
                'other.wav': os.path.join(s4_folders[3], 'other.wav')
            }
            save_mix(s4_stems, use_noise=True, filename=f"set4_mashnoisy_{i:03d}.wav")

            # ---------------------------------------------------------
            # SET 5: Cross-Genre Mashup (Dominant Genre Label)
            # ---------------------------------------------------------
            # Vocals and Bass carry the "Dominant" genre identity (Target Label)
            s5_dom_folders = random.sample(songs, 2)
            
            # Pick 2 completely different random genres for Drums and Other
            other_genres = [g for g in GENRES if g != genre]
            cross_g1, cross_g2 = random.sample(other_genres, 2)
            s5_cross_f1 = random.choice(valid_stems[cross_g1])
            s5_cross_f2 = random.choice(valid_stems[cross_g2])
            
            s5_stems = {
                'vocals.wav': os.path.join(s5_dom_folders[0], 'vocals.wav'), # Dominant
                'bass.wav': os.path.join(s5_dom_folders[1], 'bass.wav'),     # Dominant
                'drums.wav': os.path.join(s5_cross_f1, 'drums.wav'),         # Cross-Genre
                'other.wav': os.path.join(s5_cross_f2, 'other.wav')          # Cross-Genre
            }
            save_mix(s5_stems, use_noise=True, filename=f"set5_crossgenre_{i:03d}.wav")

# --- 3. EXECUTE PIPELINE ---
valid_stems, valid_noise = validate_data(STEMS_PATH, NOISE_PATH)
generate_5_set_dataset(valid_stems, valid_noise, OUTPUT_PATH, samples_per_set=SAMPLES_PER_SET)

print("\n✅ 5-Set Curriculum Dataset Successfully Generated!")

In [ ]:
# ==============================================================================
# 1. INSTALLATIONS & IMPORTS
# ==============================================================================
# Run this once in a cell: 



# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. CONFIGURATION & PATHS
# ==============================================================================
# Adjust paths based on Kaggle environment
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
TEST_CSV = os.path.join(INPUT_BASE, 'test.csv')
TEST_AUDIO_DIR = os.path.join(INPUT_BASE, 'mashups')

GENRES = sorted(["blues", "classical", "country", "disco", "hiphop", 
                 "jazz", "metal", "pop", "reggae", "rock"])
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# Audio Parameters
SR = 16000
DURATION = 10
TARGET_SAMPLES = SR * DURATION

# ==============================================================================
# 3. DATASET: DYNAMIC MASHUP GENERATION (THE SOLUTION TO THE DISTRIBUTION GAP)
# ==============================================================================


# --- 1. DATASET CLEANING: REMOVE CORRUPTED FILES ---
def validate_and_clean_recipes(stems_path, genres, target_sr=16000, min_duration=1.0):
    """
    Scans the dataset for corrupted, unreadable, or completely silent audio files.
    Returns a clean list of valid song folders (recipes).
    """
    print("Starting Dataset Validation & Cleaning...")
    valid_recipes = []
    corrupted_count = 0
    silent_count = 0
    
    for genre in genres:
        song_folders = glob.glob(os.path.join(stems_path, genre, '*'))
        
        for folder in song_folders:
            is_valid = True
            
            # Check all 4 expected stems in the folder
            for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
                stem_path = os.path.join(folder, stem)
                
                if not os.path.exists(stem_path):
                    is_valid = False
                    break
                    
                try:
                    # Attempt to load the file
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Check for pure silence (Zero Variance/Amplitude)
                    if torch.max(torch.abs(waveform)) == 0.0:
                        silent_count += 1
                        is_valid = False
                        break
                        
                    # Check if file is absurdly short
                    if waveform.shape[1] < (target_sr * min_duration):
                        is_valid = False
                        break
                        
                except Exception as e:
                    corrupted_count += 1
                    is_valid = False
                    break
            
            if is_valid:
                valid_recipes.append({'genre': genre, 'folder': folder})
                
    print(f"Validation Complete! Found {len(valid_recipes)} perfectly valid multi-stem folders.")
    print(f"Removed {corrupted_count} corrupted files and {silent_count} silent files.")
    return valid_recipes


# --- 2. VISUALIZATION: PADDING, MIXING, AND SPECTROGRAMS ---
def visualize_mashup_creation(valid_recipes, target_samples=160000):
    """
    Takes one valid recipe, pads/truncates the stems, mixes them, 
    and plots the visual transformation.
    """
    # Grab a random song
    recipe = np.random.choice(valid_recipes)
    folder = recipe['folder']
    genre = recipe['genre']
    
    stems_audio = {}
    mix = torch.zeros((1, target_samples))
    
    print(f"Visualizing Mashup Creation for Genre: {genre.upper()}")
    
    # 1. Load, Pad/Truncate, and sum the stems
    for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
        path = os.path.join(folder, stem)
        waveform, sr = torchaudio.load(path)
        
        # Resample if needed
        if sr != 16000:
            waveform = T.Resample(sr, 16000)(waveform)
            
        # Convert to Mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # The Padding / Truncation Logic
        if waveform.shape[1] > target_samples:
            waveform = waveform[:, :target_samples] # Truncate
        else:
            # Zero-Pad the end of the audio
            waveform = torch.nn.functional.pad(waveform, (0, target_samples - waveform.shape[1]))
            
        stems_audio[stem] = waveform.squeeze().numpy()
        mix += waveform
        
    # Normalize the final mix to prevent clipping [-1.0, 1.0]
    mix = mix / (torch.max(torch.abs(mix)) + 1e-9)
    mix_np = mix.squeeze().numpy()
    
    # Generate Mel-Spectrogram for the Mix
    mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128)
    to_db = T.AmplitudeToDB()
    mel_spec_db = to_db(mel_transform(mix)).squeeze().numpy()

    # --- PLOTTING ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Plot 1: Individual Padded Stems
    axes[0].set_title("1. Individual Isolated Stems (Padded/Truncated to exact length)")
    axes[0].plot(stems_audio['drums.wav'], alpha=0.7, label='Drums', color='blue')
    axes[0].plot(stems_audio['vocals.wav'], alpha=0.6, label='Vocals', color='green')
    axes[0].plot(stems_audio['bass.wav'], alpha=0.5, label='Bass', color='red')
    axes[0].legend(loc='upper right')
    axes[0].set_xlim([0, target_samples])
    
    # Plot 2: Final Summed Mashup Waveform
    axes[1].set_title(f"2. Final Normalized Mashup Waveform (Genre: {genre})")
    librosa.display.waveshow(mix_np, sr=16000, ax=axes[1], color='purple')
    
    # Plot 3: The Mel-Spectrogram (What the CNN sees)
    axes[2].set_title("3. Mel-Spectrogram of the Mashup")
    img = librosa.display.specshow(mel_spec_db, sr=16000, hop_length=512, x_axis='time', y_axis='mel', ax=axes[2], cmap='magma')
    fig.colorbar(img, ax=axes[2], format="%+2.f dB")
    
    plt.tight_layout()
    plt.show()

# ==============================================================================
# EXECUTE THE CLEANING AND VISUALIZATION
# ==============================================================================
# 1. Clean the dataset to get 'all_valid_recipes'
all_valid_recipes = validate_and_clean_recipes(STEMS_PATH, GENRES, target_sr=SR)

# 2. Visualize the math and padding
if all_valid_recipes:
    visualize_mashup_creation(all_valid_recipes, target_samples=TARGET_SAMPLES)
else:
    print("Error: No valid audio files found. Check your file paths.")

# NOTE: After running this, pass `all_valid_recipes` into your train_test_split 
# instead of the unvalidated `all_recipes` from the previous script!

class DynamicMashupDataset(Dataset):
    def __init__(self, recipes, stems_path, noise_path, mode='crnn'):
        """
        mode: 'cnn'/'crnn' for Mel-Spectrograms, 'ast' for raw waveforms
        """
        self.recipes = recipes
        self.stems_path = stems_path
        self.mode = mode
        
        # Load Noise files
        self.noise_files = glob.glob(os.path.join(noise_path, '**', '*.wav'), recursive=True)
        
        # Audio Processing
        self.mel_transform = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
        self.to_db = T.AmplitudeToDB()
        
        # For AST
        if mode == 'ast':
            self.ast_extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

    def __len__(self):
        return len(self.recipes)
    
    def _load_and_pad(self, path):
        waveform, sr = torchaudio.load(path)
        if sr != SR:
            waveform = T.Resample(sr, SR)(waveform)
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] > TARGET_SAMPLES:
            waveform = waveform[:, :TARGET_SAMPLES]
        else:
            waveform = torch.nn.functional.pad(waveform, (0, TARGET_SAMPLES - waveform.shape[1]))
        return waveform

    def __getitem__(self, idx):
        recipe = self.recipes[idx]
        genre = recipe['genre']
        label = GENRE_TO_IDX[genre]
        song_folder = recipe['folder']
        
        # 1. Mix Stems
        mix = torch.zeros((1, TARGET_SAMPLES))
        for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
            stem_p = os.path.join(song_folder, stem)
            if os.path.exists(stem_p):
                mix += self._load_and_pad(stem_p)
                
        # 2. Inject ESC-50 Noise
        if self.noise_files:
            noise_file = random.choice(self.noise_files)
            noise_wave = self._load_and_pad(noise_file)
            mix += (noise_wave * random.uniform(0.1, 0.3))
            
        # Normalize to prevent clipping
        mix = mix / (torch.max(torch.abs(mix)) + 1e-9)

        # 3. Format based on Model Architecture
        if self.mode in ['cnn', 'crnn']:
            mel = self.mel_transform(mix)
            mel_db = self.to_db(mel)
            return mel_db, label
        elif self.mode == 'ast':
            # AST expects numpy 1D array
            mix_np = mix.squeeze(0).numpy()
            inputs = self.ast_extractor(mix_np, sampling_rate=SR, return_tensors="pt")
            return inputs['input_values'].squeeze(0), label

# Generate Recipes
all_recipes = []
for genre in GENRES:
    folders = glob.glob(os.path.join(STEMS_PATH, genre, '*'))
    for folder in folders:
        all_recipes.append({'genre': genre, 'folder': folder})

# Train/Val Split
train_recipes, val_recipes = train_test_split(all_recipes, test_size=0.2, random_state=42)

# ==============================================================================
# 4. MODEL ARCHITECTURES
# ==============================================================================
# Model 1: Simple Baseline CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# Model 2: Convolutional Recurrent Neural Network (CRNN)
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        # Input size = 64 channels * 32 Mel bins = 2048
        self.lstm = nn.LSTM(input_size=2048, hidden_size=64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f) # Prepare for RNN
        lstm_out, _ = self.lstm(x)
        pooled, _ = torch.max(lstm_out, dim=1) # Global Max Pooling over time
        return self.fc(pooled)

# Model 3: AST initialization function
def get_ast_model():
    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=10,
        ignore_mismatched_sizes=True
    )
    return model

# ==============================================================================
# 5. TRAINING ENGINE WITH WANDB
# ==============================================================================
def train_model(model_name, epochs=10, batch_size=32, lr=1e-3):
    # Setup Data
    mode = 'ast' if model_name == 'ast' else 'crnn'
    train_ds = DynamicMashupDataset(train_recipes, STEMS_PATH, NOISE_PATH, mode=mode)
    val_ds = DynamicMashupDataset(val_recipes, STEMS_PATH, NOISE_PATH, mode=mode)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Initialize Model
    if model_name == 'cnn': model = SimpleCNN().to(device)
    elif model_name == 'crnn': model = CRNN().to(device)
    elif model_name == 'ast': model = get_ast_model().to(device); lr = 5e-5 # AST needs lower LR
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    
    # WANDB Initialization
    wandb.init(entity="21f2000660-dl-gen-ai-project-26-t1",project="21f2000660-t12026",name=f"{model_name}_run")
    
    best_f1 = 0.0
    
    print(f"\n--- Starting Training for {model_name.upper()} ---")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            if model_name == 'ast':
                outputs = model(inputs, labels=labels)
                loss = outputs.loss
                logits = outputs.logits
            else:
                logits = model(inputs)
                loss = criterion(logits, labels)
                
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            
        # Validation
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                if model_name == 'ast':
                    outputs = model(inputs, labels=labels)
                    loss = outputs.loss
                    logits = outputs.logits
                else:
                    logits = model(inputs)
                    loss = criterion(logits, labels)
                    
                val_loss += loss.item() * inputs.size(0)
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        # Metrics
        epoch_t_loss = train_loss / len(train_ds)
        epoch_v_loss = val_loss / len(val_ds)
        mac_f1 = f1_score(all_labels, all_preds, average='macro')
        acc = accuracy_score(all_labels, all_preds)
        
        wandb.log({"Train Loss": epoch_t_loss, "Val Loss": epoch_v_loss, "Macro F1": mac_f1, "Accuracy": acc})
        print(f"Epoch {epoch+1} | T-Loss: {epoch_t_loss:.4f} | V-Loss: {epoch_v_loss:.4f} | F1: {mac_f1:.4f}")
        
        if mac_f1 > best_f1:
            best_f1 = mac_f1
            torch.save(model.state_dict(), f"best_{model_name}.pth")
            print("⭐️ Model saved!")
            
    wandb.finish()
    return f"best_{model_name}.pth"

# ==============================================================================
# 6. EXECUTION & INFERENCE PIPELINE
# ==============================================================================
# 1. Train the model (Change 'crnn' to 'cnn' or 'ast' to try different architectures)
CHOSEN_MODEL = 'ast' 
best_model_path = train_model(model_name=CHOSEN_MODEL, epochs=10, batch_size=32)

# 2. Inference & Submission Setup
print("\n--- Generating Predictions on Test Set ---")
test_df = pd.read_csv(TEST_CSV)

# Load best model
if CHOSEN_MODEL == 'cnn': model = SimpleCNN().to(device)
elif CHOSEN_MODEL == 'crnn': model = CRNN().to(device)
elif CHOSEN_MODEL == 'ast': model = get_ast_model().to(device)

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

mel_transform = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
to_db = T.AmplitudeToDB()
ast_ext = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

predictions = []
with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_name = row['filename'] if 'filename' in test_df.columns else f"{row['id']:04d}.wav"
        file_path = os.path.join(TEST_AUDIO_DIR, file_name)
        
        # Audio Preprocessing for Test
        waveform, sr = torchaudio.load(file_path)
        if sr != SR: waveform = T.Resample(sr, SR)(waveform)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] > TARGET_SAMPLES: waveform = waveform[:, :TARGET_SAMPLES]
        else: waveform = torch.nn.functional.pad(waveform, (0, TARGET_SAMPLES - waveform.shape[1]))
        waveform = waveform / (torch.max(torch.abs(waveform)) + 1e-9)
        
        # Forward Pass
        if CHOSEN_MODEL in ['cnn', 'crnn']:
            mel_db = to_db(mel_transform(waveform)).unsqueeze(0).to(device)
            logits = model(mel_db)
        else:
            inputs = ast_ext(waveform.squeeze(0).numpy(), sampling_rate=SR, return_tensors="pt")
            inputs = inputs['input_values'].to(device)
            logits = model(inputs).logits
            
        pred_idx = torch.argmax(logits, dim=1).item()
        predictions.append(IDX_TO_GENRE[pred_idx])

# 3. Create Submission File
submission_df = pd.DataFrame({'id': test_df['id'], 'genre': predictions})
submission_df.to_csv('submission.csv', index=False)
print("\n✅ Success! submission.csv generated.")

# Complete Code

In [ ]:
# ==============================================================================
# 1. INSTALLATIONS & SETUP
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. CONFIGURATION & PATHS
# ==============================================================================
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
TEST_CSV = os.path.join(INPUT_BASE, 'test.csv')
TEST_AUDIO_DIR = os.path.join(INPUT_BASE, 'mashups')
SYNTHETIC_DIR = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

GENRES = sorted(["blues", "classical", "country", "disco", "hiphop", 
                 "jazz", "metal", "pop", "reggae", "rock"])
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

SR = 16000
DURATION = 10
TARGET_SAMPLES = SR * DURATION
SAMPLES_PER_SET = 25  # Reduced for demonstration (25*5 sets = 125 per genre = 1250 total)

# ==============================================================================
# 3. DATA VALIDATION & VISUALIZATION
# ==============================================================================
def validate_data(stems_path, noise_path):
    print("Validating all stems and noise files...")
    valid_stems = {g: [] for g in GENRES}
    
    for genre in GENRES:
        folders = glob.glob(os.path.join(stems_path, genre, '*'))
        for folder in folders:
            is_valid = True
            for stem in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']:
                p = os.path.join(folder, stem)
                if not os.path.exists(p) or os.path.getsize(p) < 1000:
                    is_valid = False; break
            if is_valid: valid_stems[genre].append(folder)
            
    raw_noise = glob.glob(os.path.join(noise_path, '**', '*.wav'), recursive=True)
    valid_noise = [n for n in raw_noise if os.path.getsize(n) > 1000]
    return valid_stems, valid_noise

def visualize_spectrogram(audio_tensor, title="Mel Spectrogram"):
    mel_transform = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
    to_db = T.AmplitudeToDB()
    mel_spec_db = to_db(mel_transform(audio_tensor)).squeeze().numpy()
    
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(mel_spec_db, sr=SR, hop_length=512, x_axis='time', y_axis='mel', cmap='magma')
    plt.colorbar(format="%+2.f dB")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# ==============================================================================
# 4. TEMPO-SYNCED CURRICULUM DATA GENERATION
# ==============================================================================
def tempo_sync_and_mix(stems_dict, noise_path=None):
    loaded_stems = {}
    for stem_name, path in stems_dict.items():
        y, _ = librosa.load(path, sr=SR)
        loaded_stems[stem_name] = y
        
    master_tempo, _ = librosa.beat.beat_track(y=loaded_stems.get('drums.wav', loaded_stems['vocals.wav']), sr=SR)
    master_tempo = float(master_tempo)
    mix = np.zeros(TARGET_SAMPLES)
    
    for stem_name, y in loaded_stems.items():
        if stem_name != 'drums.wav' and master_tempo > 0:
            stem_tempo, _ = librosa.beat.beat_track(y=y, sr=SR)
            stem_tempo = float(stem_tempo)
            if stem_tempo > 0:
                rate = np.clip(master_tempo / stem_tempo, 0.7, 1.3)
                y = librosa.effects.time_stretch(y, rate=rate)
        
        y = y[:TARGET_SAMPLES] if len(y) > TARGET_SAMPLES else np.pad(y, (0, TARGET_SAMPLES - len(y)))
        mix += y
        
    if noise_path:
        n_wave, _ = librosa.load(noise_path, sr=SR)
        n_wave = n_wave[:TARGET_SAMPLES] if len(n_wave) > TARGET_SAMPLES else np.pad(n_wave, (0, TARGET_SAMPLES - len(n_wave)))
        mix += (n_wave * random.uniform(0.1, 0.3))
        
    mix = mix / (np.max(np.abs(mix)) + 1e-9) # Normalize
    return torch.tensor(mix, dtype=torch.float32).unsqueeze(0)

def build_dataset(valid_stems, valid_noise):
    print("Building Tempo-Synced 5-Set Curriculum Dataset... (This takes time)")
    os.makedirs(SYNTHETIC_DIR, exist_ok=True)
    generated_files = []
    
    for genre in GENRES:
        genre_out = os.path.join(SYNTHETIC_DIR, genre)
        os.makedirs(genre_out, exist_ok=True)
        songs = valid_stems[genre]
        if len(songs) < 4: continue
            
        for i in range(SAMPLES_PER_SET):
            # Set 1 & 2: Clean and Noisy (Same stems)
            base_folder = random.choice(songs)
            base_stems = {s: os.path.join(base_folder, s) for s in ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']}
            
            mix1 = tempo_sync_and_mix(base_stems, noise_path=None)
            p1 = os.path.join(genre_out, f"set1_{i:03d}.wav")
            torchaudio.save(p1, mix1, SR); generated_files.append({'path': p1, 'genre': genre})
            
            mix2 = tempo_sync_and_mix(base_stems, noise_path=random.choice(valid_noise))
            p2 = os.path.join(genre_out, f"set2_{i:03d}.wav")
            torchaudio.save(p2, mix2, SR); generated_files.append({'path': p2, 'genre': genre})
            
            # (In a real run, you would add Sets 3, 4, and 5 here as previously described)
            
    return generated_files

# ==============================================================================
# 5. PYTORCH DATASET LOADING
# ==============================================================================
class MashupDiskDataset(Dataset):
    def __init__(self, file_list, mode='cnn'):
        self.file_list = file_list
        self.mode = mode
        self.mel_tf = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
        self.to_db = T.AmplitudeToDB()
        if mode == 'ast':
            self.ast_extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]['path']
        label = GENRE_TO_IDX[self.file_list[idx]['genre']]
        waveform, _ = torchaudio.load(path) # Already mono & 160k samples from build phase
        
        if self.mode in ['cnn', 'crnn']:
            return self.to_db(self.mel_tf(waveform)), label
        else:
            inputs = self.ast_extractor(waveform.squeeze(0).numpy(), sampling_rate=SR, return_tensors="pt")
            return inputs['input_values'].squeeze(0), label

# ==============================================================================
# 6. MODEL ARCHITECTURES
# ==============================================================================
class EfficientNetAudio(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.efnet = models.efficientnet_b0(weights=None)
        orig_conv = self.efnet.features[0][0]
        self.efnet.features[0][0] = nn.Conv2d(1, orig_conv.out_channels, orig_conv.kernel_size, 
                                              orig_conv.stride, orig_conv.padding, bias=False)
        self.efnet.classifier[1] = nn.Linear(self.efnet.classifier[1].in_features, num_classes)
    def forward(self, x): return self.efnet(x)

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.lstm = nn.LSTM(2048, 64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.cnn(x)
        b, c, f, t = x.shape
        lstm_out, _ = self.lstm(x.permute(0, 3, 1, 2).reshape(b, t, c * f))
        pooled, _ = torch.max(lstm_out, dim=1)
        return self.fc(pooled)

def get_ast_model():
    return ASTForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", 
                                                     num_labels=10, ignore_mismatched_sizes=True)

# ==============================================================================
# 7. TRAINING ENGINE
# ==============================================================================
def train_pipeline(model_name='cnn', epochs=5, batch_size=16):
    valid_stems, valid_noise = validate_data(STEMS_PATH, NOISE_PATH)
    all_files = build_dataset(valid_stems, valid_noise)
    
    # Visualization Check
    if all_files:
        sample_wave, _ = torchaudio.load(all_files[0]['path'])
        visualize_spectrogram(sample_wave, title=f"Sample Spec: {all_files[0]['genre']}")
    
    train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
    mode = 'ast' if model_name == 'ast' else model_name
    
    train_loader = DataLoader(MashupDiskDataset(train_files, mode), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(MashupDiskDataset(val_files, mode), batch_size=batch_size, shuffle=False)
    
    if model_name == 'cnn': model = EfficientNetAudio().to(device); lr = 1e-3
    elif model_name == 'crnn': model = CRNN().to(device); lr = 1e-3
    elif model_name == 'ast': model = get_ast_model().to(device); lr = 5e-5
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    
    wandb.init(project="Messy_Mashup", name=f"{model_name}_training")
    best_f1 = 0.0
    
    for epoch in range(epochs):
        model.train(); train_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            if model_name == 'ast':
                outputs = model(inputs, labels=labels)
                loss = outputs.loss
            else:
                loss = criterion(model(inputs), labels)
            loss.backward(); optimizer.step(); train_loss += loss.item()
            
        model.eval(); all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits = model(inputs).logits if model_name == 'ast' else model(inputs)
                all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        mac_f1 = f1_score(all_labels, all_preds, average='macro')
        wandb.log({"Train Loss": train_loss/len(train_loader), "Val F1": mac_f1})
        print(f"Epoch {epoch+1} | Val F1: {mac_f1:.4f}")
        
        if mac_f1 > best_f1:
            best_f1 = mac_f1
            torch.save(model.state_dict(), f"/kaggle/working/best_{model_name}.pth")
            
    wandb.finish()
    return f"/kaggle/working/best_{model_name}.pth"

# ==============================================================================
# 8. EXECUTE TRAINING & INFERENCE
# ==============================================================================
# Choose 'cnn' (EfficientNet), 'crnn', or 'ast'
CHOSEN_MODEL = 'cnn' 

# 1. Run Training
best_weights = train_pipeline(model_name=CHOSEN_MODEL, epochs=5)

# 2. Run Inference on Test Set
print("Running Inference...")
test_df = pd.read_csv(TEST_CSV)
if CHOSEN_MODEL == 'cnn': model = EfficientNetAudio().to(device)
elif CHOSEN_MODEL == 'crnn': model = CRNN().to(device)
elif CHOSEN_MODEL == 'ast': model = get_ast_model().to(device)

model.load_state_dict(torch.load(best_weights, map_location=device))
model.eval()

mel_tf = T.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=512, n_mels=128)
to_db = T.AmplitudeToDB()
ast_ext = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

preds = []
with torch.no_grad():
    for _, row in test_df.iterrows():
        path = os.path.join(TEST_AUDIO_DIR, f"{row['id']:04d}.wav")
        wave, sr = torchaudio.load(path)
        if sr != SR: wave = T.Resample(sr, SR)(wave)
        if wave.shape[0] > 1: wave = torch.mean(wave, dim=0, keepdim=True)
        wave = wave[:, :TARGET_SAMPLES] if wave.shape[1] > TARGET_SAMPLES else torch.nn.functional.pad(wave, (0, TARGET_SAMPLES - wave.shape[1]))
        wave = wave / (torch.max(torch.abs(wave)) + 1e-9)
        
        if CHOSEN_MODEL in ['cnn', 'crnn']:
            logits = model(to_db(mel_tf(wave)).unsqueeze(0).to(device))
        else:
            inputs = ast_ext(wave.squeeze(0).numpy(), sampling_rate=SR, return_tensors="pt")
            logits = model(inputs['input_values'].to(device)).logits
            
        preds.append(IDX_TO_GENRE[torch.argmax(logits, dim=1).item()])

pd.DataFrame({'id': test_df['id'], 'genre': preds}).to_csv('/kaggle/working/submission.csv', index=False)
print("✅ submission.csv saved in /kaggle/working!")

# DummyModel Submission for Registration

In [ ]:
if execute_milestone_solutions_code:
    
    testData = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
    path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
    
    with os.scandir(path) as entries:
        directories = [entry.name for entry in entries]
    
    ids = [f"{i:04d}" for i in range(1, len(testData)+1)]
    genres = random.choices(directories, k=3020)
    
    submission = pd.DataFrame({
        'id': ids,
        'genre' : genres
    })
    
    submission.head()
    
    submission.to_csv('/kaggle/working/submission.csv', index=False)

# Submission using the DecisionTreeClassifier model

In [ ]:
TEST_CSV_PATH = os.path.join(ROOT, 'test.csv')

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(song_path, sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

test_df = pd.read_csv(TEST_CSV_PATH)

#print(test_df.shape)
#print(test_df.sample(5))

X_test = []

for index, row in test_df.iterrows():
    file_name = row['filename']
    file_path = os.path.join(ROOT, file_name)
    features = extract_features(file_path)
    X_test.append(features)

X_test = np.array(X_test)
test_pred = clf.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'genre' : test_pred
})

submission.head()

submission.to_csv('/kaggle/working/submission.csv', index=False)